# Phase 5: generator transfer matrix for conformal hallucination filters (Q1 extension)

## THIS COPY RUNS WAVE 2. Nothing to edit in the cells.

**Each session:** Save Version -> Save & Run All (Commit). The laptop can be switched off; it runs on Kaggle.

**From the second session on:** first Add Input -> Your Work -> this notebook (latest version), then commit again. It continues from where the last session stopped.


**Kaggle settings:** Accelerator **GPU T4 x2**, **Internet ON**, Persistence off.

**What it runs.** Six caption generators x three detectors on 1,000 + 3,000 fresh COCO images and 804 AMBER images,
then the transfer matrix, prevalence decomposition, label-free warning, recalibration curve, grouped CRC and BH
decisions (`scripts/phase5.py`). Results go to `/kaggle/working/phase5/phase5_results.zip`.

**Always run with Save Version -> Save & Run All (Commit).** That runs in the background with the browser closed.
An interactive session (the Run buttons) stops when you close the tab, and its files are lost.
Cell 6 runs for hours; cells 7 to 10 show nothing until it ends. Cell 6 prints one progress line every 10 minutes.

**Order of work**
1. **Pilot (one commit, about 1.5 h).** Set `MODE = "pilot"` in cell 2. Cell 8 prints CHAIR
   and truncation for every generator on 100 images. Check the captions look sensible before going on.
2. **Wave 1** (`MODE = "full"`, `WAVE = 1`: LLaVA, Qwen2.5-VL-3B, InternVL3-2B). About 2 commits of 11 h.
   Each session: **Save Version -> Save & Run All (Commit)**. It stops by itself after `SESSION_HOURS`.
   From the second session on: **Add Input -> Your Work -> this notebook (latest version)** first, so cell 4 imports
   the finished work.
3. **Wave 2** (`WAVE = 2`: sampled LLaVA, Qwen3-VL-4B, InstructBLIP). About 3 commits of 11 h.
4. When cell 7 says a wave is COMPLETE, cell 9 runs the analysis and the audit sample.

**Before cell 9 runs on full data, the pre-registration must be frozen** (`prereg/prereg-v3-DRAFT.md` -> `prereg-v3.md`,
git tag `prereg-v3`) and `REF` in cell 3 set to that tag.

**Quota.** Kaggle gives about 30 GPU hours a week. Wave 1 needs about 22 h of session time, wave 2 about 26 h.

In [ ]:
# 1. Environment and GPUs
import os, subprocess, sys, time, glob, json, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU FOUND - switch the accelerator to GPU T4 x2")
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU)

In [ ]:
# 2. Settings: change only these
MODE = "full"          # "pilot" (100 images per generator, separate folder) or "full"
WAVE = 2                # 1 = llava, qwen25, internvl3 ; 2 = llava_t1, qwen3_4b, instructblip
SESSION_HOURS = 11.0    # GPU time per session (Kaggle stops a session at 12 h; ~30 min left for setup and saving)
WAVES = {1: "llava,qwen25,internvl3", 2: "llava_t1,qwen3_4b,instructblip"}
GENS = WAVES[WAVE] if MODE == "full" else WAVES[1] + "," + WAVES[2]
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
OUT = os.path.join(WORK, "phase5" if MODE == "full" else "phase5_pilot")
os.makedirs(OUT, exist_ok=True)
LIMIT = ["--limit", "100"] if MODE == "pilot" else []
print(MODE, "| generators:", GENS, "| results:", OUT)

In [ ]:
# 3. Code and packages. After the pre-registration is frozen, set REF = "prereg-v3".
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
REF = "q1-tai-upgrade"
os.chdir(WORK)
if not os.path.exists("bcrbp"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO, "bcrbp"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!git log --oneline -1

In [ ]:
# 4. Setup (annotations, the seeded Phase 5 split), import of earlier sessions, images (CPU, ~15-25 min)
!python scripts/phase5.py setup  --out "$OUT"
!python scripts/phase5.py import --out "$OUT"
!python scripts/phase5.py images --out "$OUT"
p = json.load(open(os.path.join(OUT, "prep5.json")))
print("calibration", len(p["cal"]), "| test", len(p["test"]), "| AMBER", len(p["amber"]))

In [ ]:
# 5. Smoke test: 2 images per generator in a scratch folder (skipped once results exist)
SMOKE = os.path.join(WORK, "phase5_smoke")
if glob.glob(os.path.join(OUT, "captions5_*.s*.jsonl")):
    print("Skipping the smoke test: results already exist.")
else:
    os.makedirs(os.path.join(SMOKE, "amber"), exist_ok=True)
    for f in ["synonyms.txt", "prep5.json", "prep.json"]:
        shutil.copy(os.path.join(OUT, f), os.path.join(SMOKE, f))
    shutil.copy(os.path.join(OUT, "amber", "annotations.json"), os.path.join(SMOKE, "amber", "annotations.json"))
    for src, dst in [(os.path.join(OUT, "images"), os.path.join(SMOKE, "images")),
                     (os.path.join(OUT, "amber", "images"), os.path.join(SMOKE, "amber", "images"))]:
        if not os.path.exists(dst):
            os.symlink(src, dst)
    !python scripts/phase5.py captions --out "$SMOKE" --gens "$GENS" --limit 2
    !python scripts/phase5.py detect   --out "$SMOKE" --gens "$GENS"
    for f in sorted(glob.glob(os.path.join(SMOKE, "captions5_*_coco.s0.jsonl"))):
        r = json.loads(open(f).readline())
        print(os.path.basename(f), "->", r["caption"][:160])
    print("OWLv2 rows:", sum(1 for f in glob.glob(os.path.join(SMOKE, "owl5_*.jsonl")) for _ in open(f)))

In [ ]:
# 6. GPU work for THIS session: one process per GPU (captions, then detectors), stopped after SESSION_HOURS
#    A progress report with the time left is printed every 10 minutes.
# Progress display: rows finished for each model, % of this wave done, estimated time left
import math
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()          # clock times in India time
sys.path.insert(0, "scripts")
import phase5 as P5, phase4 as P4
from pathlib import Path
SEC = {"llava": 9, "llava_t1": 9, "qwen25": 15, "qwen3_4b": 18, "internvl3": 7, "instructblip": 9}  # s per caption, T4
DET_SEC = 2.0                                   # s per image for the three detectors together (estimate)
_p = P5._prep(Path(OUT))
WANT = {"coco": (len(_p["cal"]) + len(_p["test"])) if MODE == "full" else 100,
        "amber": len(_p["amber"]) if MODE == "full" else 100}

def _counts():
    c = {}
    for g in GENS.split(","):
        for ds in WANT:
            caps = len(P4.read_all(Path(OUT), P5.cap_file(g, ds)))
            det = min(len(P4.read_all(Path(OUT), f"owl5_{g}_{ds}")), len(P4.read_all(Path(OUT), f"swap5_{g}_{ds}")))
            c[g, ds] = (caps, det)
    return c

def _work(c):
    """GPU-seconds of finished and of total work, at the T4 speeds above."""
    done = total = 0.0
    for (g, ds), (caps, det) in c.items():
        w = WANT[ds]
        done += min(caps, w) * SEC.get(g, 10) + min(det, w) * DET_SEC
        total += w * (SEC.get(g, 10) + DET_SEC)
    return done, total

_W0 = _work(_counts())[0]                      # work already finished when this session started

def show_progress(t0, ngpu):
    c = _counts()
    done, total = _work(c)
    el = time.time() - t0
    speed = el * ngpu / (done - _W0) if done - _W0 > 600 else 1.0   # measured speed once enough is done
    left_h = (total - done) * speed / ngpu / 3600
    session_left = max(0.0, SESSION_HOURS - el / 3600)
    print(f"\n{time.strftime('%H:%M')} | running {el / 3600:.1f} h | this session stops in {session_left:.1f} h")
    w = sum(WANT.values())
    for g in GENS.split(","):
        caps = sum(c[g, ds][0] for ds in WANT)
        det = sum(c[g, ds][1] for ds in WANT)
        print(f"  {g:13s} captions {caps:5d}/{w} ({100 * caps / w:3.0f}%)   detectors {det:5d}/{w} ({100 * det / w:3.0f}%)")
    more = math.ceil((left_h - session_left) / SESSION_HOURS) if left_h > session_left else 0
    print(f"  WAVE {100 * done / total:.0f}% done | about {left_h:.1f} h of work left"
          + (f" | about {more} more session(s) after this one" if more else " | should finish in this session"),
          flush=True)
procs = []
for k in range(max(1, NGPU)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    sh = (f'python scripts/phase5.py captions --out "{OUT}" --gens {GENS} --shard {k} --nshards {max(1, NGPU)} {" ".join(LIMIT)} && '
          f'python scripts/phase5.py detect --out "{OUT}" --gens {GENS} --shard {k} --nshards {max(1, NGPU)}')
    log = open(os.path.join(OUT, f"gpu5_{k}.log"), "a")
    procs.append(subprocess.Popen(["bash", "-c", sh], stdout=log, stderr=subprocess.STDOUT, env=env))
t0 = time.time()
show_progress(t0, max(1, NGPU))
shown = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    if time.time() - shown > 600:                          # every 10 min
        show_progress(t0, max(1, NGPU))
        for k in range(max(1, NGPU)):
            tail = open(os.path.join(OUT, f"gpu5_{k}.log")).readlines()[-1:]
            print(f"  GPU {k} last log line:", tail[0].strip() if tail else "starting", flush=True)
        shown = time.time()
    if time.time() - t0 > SESSION_HOURS * 3600:
        print("Session time limit reached: stopping (finished items are saved).")
        for p in procs:
            if p.poll() is None:
                p.terminate()
        break
print("GPU work stopped after", round((time.time() - t0) / 3600, 2), "h; exit codes:", [p.poll() for p in procs])
show_progress(t0, max(1, NGPU))
for k in range(max(1, NGPU)):
    print(f"--- gpu5_{k}.log (last lines)")
    print("".join(open(os.path.join(OUT, f"gpu5_{k}.log")).readlines()[-6:]))

In [ ]:
# 7. Progress check
sys.path.insert(0, "scripts")
import importlib, phase5 as P5, phase4 as P4
from pathlib import Path
importlib.reload(P5)
OUTP = Path(OUT)
p = P5._prep(OUTP)
n_coco = len(p["cal"]) + len(p["test"]) if MODE == "full" else 100
n_amber = len(p["amber"]) if MODE == "full" else 100
COMPLETE = True
for g in GENS.split(","):
    for ds, want in (("coco", n_coco), ("amber", n_amber)):
        caps = len(P4.read_all(OUTP, P5.cap_file(g, ds)))
        owl = len(P4.read_all(OUTP, f"owl5_{g}_{ds}"))
        swap = len(P4.read_all(OUTP, f"swap5_{g}_{ds}"))
        ok = caps >= want and owl >= 0.97 * caps and swap >= 0.97 * caps
        COMPLETE &= ok
        print(f"{g:13s} {ds:6s} captions {caps:5d}/{want:<5d} owl {owl:5d} swap {swap:5d} {'done' if ok else ''}")
print("COMPLETE" if COMPLETE else "Not complete: run another session.")

In [ ]:
# 8. Pilot check: CHAIR and truncation per generator (Gate 1)
!python scripts/phase5.py pilot --out "$OUT" --gens $GENS

In [ ]:
# 9. Analysis and audit sample (CPU). Runs only when the wave is complete.
#    On full data this must run from the frozen pre-registration (REF = "prereg-v3").
ALL = ",".join(g for g in WAVES[1].split(",") + WAVES[2].split(",")
               if P4.read_all(OUTP, P5.cap_file(g, "coco")))
if MODE == "pilot":
    print("Pilot mode: no analysis (the pilot images are calibration images). Check cell 8 instead.")
elif COMPLETE:
    if REF != "prereg-v3":
        print("Not running the confirmatory analysis: freeze the pre-registration and set REF = 'prereg-v3' first.")
    else:
        !python scripts/phase5.py analyze --out "$OUT" --gens "$ALL"
        !python scripts/phase5.py audit --out "$OUT" --gens "llava,qwen25,internvl3"
        r = json.load(open(os.path.join(OUT, "phase5_results.json")))
        for s, row in r["matrices"]["owlv2"].items():
            print(s, {t: (round(c["claim"]["point"], 3), c["claim"]["decision"][:4]) for t, c in row.items()})

In [ ]:
# 10. Package results (no images); made every session
import zipfile
zpath = os.path.join(OUT, "phase5_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in (glob.glob(f"{OUT}/*.jsonl") + glob.glob(f"{OUT}/*.log") + glob.glob(f"{OUT}/audit/*.csv")
              + [f"{OUT}/prep5.json", f"{OUT}/phase5_results.json"]):
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, OUT))
    z.writestr("environment.txt", f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("Saved:", zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB |", "COMPLETE" if COMPLETE else "partial")